In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.market_math import _prepare_pair_parameters
from src.market_math import compute_log_spread
from src.market_math import _stable_seed
from src.convergence_signal import calculate_convergence_signal


# 04 Conditional fOU Convergence Signal

Inspect a single-date signal snapshot with the same forecast function used by the daily backtest. This preview is not the full strategy signal history: Module 07 recalculates forecasts daily on flat pairs.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Evaluation date and frozen parameters

The first test session is the default preview date. The complete backtest does not depend on this preview date.


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
full_prices = pd.concat([train_prices, test_prices])
params = _prepare_pair_parameters(
    pd.read_parquet("eligible_pairs.parquet"),
    pd.read_parquet("cointegrated_pairs.parquet"),
).sort_values("pair")
EVALUATION_DATE = test_prices.index[0]
print("Signal snapshot date:", EVALUATION_DATE.date())


## 3. Conditional forecasts

Forecasts use observations through the evaluation close only; a qualifying order could execute at the following close.


In [ ]:
rows = []
curves = {}
for row in params.itertuples():
    history = compute_log_spread(
        full_prices, row.dependent, row.independent, row.alpha, row.beta
    ).loc[:EVALUATION_DATE]
    z = (history.iloc[-1] - row.mu) / np.sqrt(row.variance)
    if abs(z) < cfg.entry_z:
        continue
    sig, curve = calculate_convergence_signal(
        history,
        row.mu,
        row.kappa,
        row.sigma,
        row.hurst,
        row.variance,
        target_probability=cfg.target_probability,
        entry_z=cfg.entry_z,
        memory_window=cfg.memory_window,
        max_horizon_days=cfg.max_horizon_days,
        n_paths=cfg.n_paths,
        seed=_stable_seed(cfg.seed, row.pair, EVALUATION_DATE),
    )
    if sig.selected_dte_trading_days is None:
        continue
    rows.append(
        dict(
            pair=row.pair,
            dependent=row.dependent,
            independent=row.independent,
            beta=row.beta,
            signal_date=EVALUATION_DATE,
            direction=sig.direction,
            z=z,
            horizon=sig.selected_dte_trading_days,
            probability=sig.probability_at_selected_dte,
        )
    )
    curves[row.pair] = curve
snapshot = pd.DataFrame(
    rows,
    columns=[
        "pair",
        "dependent",
        "independent",
        "beta",
        "signal_date",
        "direction",
        "z",
        "horizon",
        "probability",
    ],
)
snapshot.to_parquet("signal_snapshot.parquet")
pd.DataFrame(curves).to_parquet("signal_probability_curves.parquet")
display(snapshot)
if curves:
    pd.DataFrame(curves).iloc[:, :5].plot(
        figsize=(10, 4), title="Conditional first passage probabilities"
    )
    plt.axhline(cfg.target_probability, color="black", linestyle="--")
    plt.show()
else:
    print(
        "No qualifying signals on this preview date. Continue to Module 05; this is a valid result."
    )
